### Tutorial 0 — Download the example dataset (cHL CODEX)

Every other CORAL tutorial runs on the same real slide: a single
**classical Hodgkin lymphoma (cHL)** region imaged by **CODEX**
(multiplexed immunofluorescence). This short tutorial downloads that
dataset once and gets it ready to feed into CORAL.

The data was released with **MAPS** ([Shaban et al., *Nature
Communications*, 2024](https://doi.org/10.1038/s41467-024-45999-1)) and
is hosted on [Zenodo](https://zenodo.org/records/10067010) under
CC-BY-4.0. The archive holds one single-channel TIFF per marker, a cell
segmentation mask, and a cell-type annotation table.

> **One-time ~3.4 GB download.** Fetch it once; every later tutorial
> reuses the result. You can launch Jupyter from either the `tutorials/`
> directory or the repo root — the first cell handles both.

What this tutorial covers:

1. **Download** the cHL CODEX dataset from Zenodo
2. **Inspect** what you got (markers, mask, annotations)

Ingesting these raw TIFFs into a canonical CORAL slide is the first step of
**Tutorial 1**, which picks up from the folder you download here.

#### 1. Download the dataset

`download_chl_maps_dataset` fetches the archive from Zenodo, unzips it,
and deletes the archive afterwards to save disk. It is **idempotent**: if
the data is already unpacked it returns instantly, so re-running this
notebook is cheap.

In [ ]:
import shutil
import sys
from pathlib import Path

# Make the tutorial's utils/ importable whether Jupyter launched from
# tutorials/ or from the repo root, and pin a deterministic data dir.
cwd = Path.cwd()
tutorials_dir = cwd if (cwd / "utils").exists() else cwd / "tutorials"
sys.path.insert(0, str(tutorials_dir))

from utils import download_chl_maps_dataset

DATA_DIR = tutorials_dir / "example-data"      # git-ignored; safe to delete

chl_dir = download_chl_maps_dataset(DATA_DIR)  # returns .../cHL_CODEX

# The `coral` CLI ingests a *directory of slides* (a cohort). Our one slide
# is the per-channel raw_image/ folder, so nest it under a cohort/ folder
# that holds nothing else — then Tutorial 1 can point
# `coral ingest --image-dir` straight at cohort/, no filtering needed.
# Idempotent: the move runs only on the first pass.
cohort_dir = chl_dir / "cohort"
raw_image_dir = cohort_dir / "raw_image"
legacy_dir = chl_dir / "raw_image"             # pre-cohort download location
if legacy_dir.exists() and not raw_image_dir.exists():
    cohort_dir.mkdir(exist_ok=True)
    shutil.move(str(legacy_dir), str(raw_image_dir))

# Name the segmentation mask after its slide so the cell pipeline can pair
# them by name: raw_image/ ingests to raw_image.zarr, and the CLI expects
# the matching mask at raw_image.tiff. Idempotent: renames only once.
mask_dir = chl_dir / "segmentation"
legacy_mask = mask_dir / "cHL_CODEX_segmentation.tiff"
slide_mask = mask_dir / "raw_image.tiff"
if legacy_mask.exists() and not slide_mask.exists():
    shutil.move(str(legacy_mask), str(slide_mask))

print("dataset root:", chl_dir)
print("cohort dir  :", cohort_dir)
print("raw images  :", raw_image_dir)
print("mask        :", slide_mask)

#### 2. What you downloaded

After the step above, the dataset is laid out like this:

| Path | Contents |
| ---- | -------- |
| `cHL_CODEX/cohort/raw_image/` | one single-channel TIFF per marker — **the image** |
| `cHL_CODEX/segmentation/` | a cell instance-segmentation mask |
| `cHL_CODEX/annotation_csv/` | per-cell centroids and cell-type labels |

The `coral` CLI ingests a **directory of slides** (a cohort), so the cell
above nested our single slide — the per-channel `raw_image/` directory —
inside a `cohort/` folder that holds nothing else. Tutorial 1 then points
`coral ingest --image-dir` straight at `cohort/`.

CORAL is **patch-centric**, so `cohort/raw_image/` is all the core pipeline
(ingest → tissue → patches → features) needs. The mask and annotations
belong to the **cell** sibling capability — they're here if you want them
later, but no step below uses them.

In [ ]:
mask_dir = chl_dir / "segmentation"
annot_dir = chl_dir / "annotation_csv"

markers = sorted(p.stem for p in raw_image_dir.glob("*.tif*"))
print(f"{len(markers)} marker channels:")
print(", ".join(markers))

print("\nsegmentation:", [p.name for p in mask_dir.glob("*.tif*")])
print("annotations :", [p.name for p in annot_dir.glob("*.csv")])

#### Where next

You now have the cHL CODEX dataset downloaded and inspected — the raw
input every other tutorial builds on.

- **Tutorial 1 — ingest.** Point it at the `cohort/` folder you just
  arranged: build and review the marker map, then ingest to a
  canonical CORAL slide.
- **Tutorial 2 — tissue → patches → features.** Segment tissue, tile
  the slide into patches, and encode each patch (mean-marker baseline
  and KRONOS2).
- **Cohorts.** `convert_to_canonical` ingests one slide; the `coral
  ingest` CLI ingests a whole directory of slides in parallel. See the
  [README](../README.md) for the cohort layout and the CLI equivalents
  of every step.
- **Cells (sibling capability).** The `segmentation/` mask and
  `annotation_csv/` labels you downloaded feed the cell-centric path
  (`slide.segment_cells(...)`, cell-centered patches) once you install
  the `cells` extra.